# Stage 3: Vector Store Creation

In this stage, we'll learn how to:
1. Convert chunks into LangChain Document objects
2. Generate embeddings using sentence-transformers (all-MiniLM-L6-v2)
3. Create FAISS indices for fast similarity search
4. Save vector stores to disk for later retrieval

Key Concept: We maintain TWO separate vector stores:
- Documentation store: Theoretical content (hierarchical chunks)
- Code store: Code examples with metadata (summaries + keywords)

This separation allows us to retrieve different types of context:
- Docs provide conceptual understanding
- Code provides implementation patterns

Output: tutorial_vectorstore_save/, tutorial_code_save/ (FAISS indices)

In [1]:
import os
import pickle
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

from config import (
    DOC_CHUNKS_PATH,
    CODE_CHUNKS_PATH,
    VECTORSTORE_DOC_PATH,
    VECTORSTORE_CODE_PATH,
    EMBEDDING_MODEL,
    NUM_SIMILARITY_TESTS,
    VERBOSE
)

## SECTION 1: CREATE FAISS VECTOR STORE

In [2]:
def create_faiss_store(documents: list, embedding_model_name: str) -> FAISS:
    """
    Create a FAISS vector store from documents.

    Pipeline:
    1. Initialize embedding model (downloads if not cached)
    2. Embed all documents (converts text → vectors)
    3. Build FAISS index (for fast similarity search)

    Args:
        documents: List of LangChain Document objects
        embedding_model_name: HuggingFace model name

    Returns:
        FAISS vector store instance

    Technical Details:
        - all-MiniLM-L6-v2: 384-dimensional embeddings, ~22M parameters
        - FAISS: Facebook AI Similarity Search (efficient nearest neighbor)
        - Index type: Flat (exact search) - suitable for datasets < 1M vectors
    """
    print(f"\nCreating FAISS vector store...")
    print(f"   Embedding model: {embedding_model_name}")
    print(f"   Documents: {len(documents)}")

    # Initialize embedding model
    embeddings = HuggingFaceEmbeddings(
        model_name=embedding_model_name,
        model_kwargs={"trust_remote_code": True}
    )

    # Create FAISS index
    print(f"Embedding documents (this may take a few minutes)...")
    faiss_store = FAISS.from_documents(documents, embeddings)

    print(f"FAISS store created successfully")
    return faiss_store


def save_faiss_store(faiss_store: FAISS, save_path: str):
    """
    Save FAISS vector store to disk.

    Saves two files:
    - index.faiss: Binary vector index
    - index.pkl: Metadata (document text, metadata)

    Args:
        faiss_store: FAISS instance
        save_path: Directory path to save
    """
    os.makedirs(save_path, exist_ok=True)
    faiss_store.save_local(save_path)
    print(f"Saved to: {save_path}")

## SECTION 2: PREPARE DOCUMENTS

In [3]:
def prepare_documentation_documents(chunks: list) -> list:
    """
    Convert documentation chunks to LangChain Document objects.

    Each Document contains:
    - page_content: The actual text to be embedded
    - metadata: Additional info (heading, level, source PDF)

    Args:
        chunks: List of chunk dictionaries from Stage 1

    Returns:
        List of Document objects

    Why page_content matters:
        This is what gets embedded! The embedding model only sees this text,
        so it should be complete and self-contained.
    """
    documents = []

    for chunk in chunks:
        doc = Document(
            page_content=chunk['content'],  # ← This gets embedded
            metadata={
                'heading_title': chunk['title'],
                'heading_level': chunk['level'],
                'source_pdf': chunk['pdf_name']
            }
        )
        documents.append(doc)

    return documents


def prepare_code_documents(chunks: list) -> list:
    """
    Convert code chunks to LangChain Document objects.

    Key Difference from Documentation:
    - page_content: Summary + keywords (NOT the raw code!)
    - metadata.original_text: The actual code (retrieved later)

    This is the HyDE (Hypothetical Document Embeddings) approach:
    We embed a natural language description, not the code itself.

    Args:
        chunks: List of enriched code chunks from Stage 2

    Returns:
        List of Document objects
    """
    documents = []

    for chunk in chunks:
        # Combine summary and keywords for embedding
        # This provides the best semantic match with natural language queries
        combined_text = f"{chunk['summary']}\n\nKeywords: {chunk['keywords']}"

        doc = Document(
            page_content=combined_text,  # ← Metadata gets embedded
            metadata={
                'original_text': chunk['content'],  # ← Original code stored here
                'heading_title': chunk['title'],
                'heading_level': chunk['level'],
                'source_pdf': chunk['pdf_name'],
                'summary': chunk['summary'],
                'keywords': chunk['keywords']
            }
        )
        documents.append(doc)

    return documents

## SECTION 3: TEST SIMILARITY SEARCH

In [4]:
def test_similarity_search(store: FAISS, query: str, k: int = 3) -> list:
    """
    Test vector store with a sample query.

    Args:
        store: FAISS vector store
        query: Natural language query
        k: Number of results to return

    Returns:
        List of (document, score) tuples
    """
    results = store.similarity_search_with_score(query, k=k)
    return results

## SECTION 4: EXECUTE PIPELINE

In [5]:
print("=" * 80)
print("STAGE 3: VECTOR STORE CREATION")
print("=" * 80)

# ========================================================================
# Part 1: Create Documentation Vector Store
# ========================================================================

print(f"\n{'─' * 80}")
print("PART 1: Documentation Vector Store")
print(f"{'─' * 80}")

# Load documentation chunks from Stage 1
with open(DOC_CHUNKS_PATH, 'rb') as f:
    doc_chunks = pickle.load(f)

print(f"Loaded {len(doc_chunks)} documentation chunks")

# Prepare documents
doc_documents = prepare_documentation_documents(doc_chunks)
print(f"Prepared {len(doc_documents)} Document objects")

# Create vector store
doc_store = create_faiss_store(doc_documents, EMBEDDING_MODEL)

# Save to disk
save_faiss_store(doc_store, VECTORSTORE_DOC_PATH)

# ========================================================================
# Part 2: Create Code Examples Vector Store
# ========================================================================

print(f"\n{'─' * 80}")
print("PART 2: Code Examples Vector Store")
print(f"{'─' * 80}")

# Load code chunks from Stage 2
with open(CODE_CHUNKS_PATH, 'rb') as f:
    code_chunks = pickle.load(f)

print(f"Loaded {len(code_chunks)} code chunks with metadata")

# Prepare documents
code_documents = prepare_code_documents(code_chunks)
print(f"Prepared {len(code_documents)} Document objects")

# Create vector store
code_store = create_faiss_store(code_documents, EMBEDDING_MODEL)

# Save to disk
save_faiss_store(code_store, VECTORSTORE_CODE_PATH)

STAGE 3: VECTOR STORE CREATION

────────────────────────────────────────────────────────────────────────────────
PART 1: Documentation Vector Store
────────────────────────────────────────────────────────────────────────────────
Loaded 20 documentation chunks
Prepared 20 Document objects

Creating FAISS vector store...
   Embedding model: all-MiniLM-L6-v2
   Documents: 20
Embedding documents (this may take a few minutes)...
FAISS store created successfully
Saved to: /Users/tasnimahmed/Downloads/tutorial/Standalone/tutorial_vectorstore_save

────────────────────────────────────────────────────────────────────────────────
PART 2: Code Examples Vector Store
────────────────────────────────────────────────────────────────────────────────
Loaded 20 code chunks with metadata
Prepared 20 Document objects

Creating FAISS vector store...
   Embedding model: all-MiniLM-L6-v2
   Documents: 20
Embedding documents (this may take a few minutes)...
FAISS store created successfully
Saved to: /Users/ta

In [6]:
print(f"\n{'=' * 80}")
print("DEMONSTRATION: Similarity Search Tests")
print(f"{'=' * 80}\n")

# Test queries representing typical user questions
test_queries = [
    "How do I create binary decision variables for optimization?",
    "What are continuous variables and how do they differ from integer variables?"
][:NUM_SIMILARITY_TESTS]

for i, query in enumerate(test_queries, 1):
    print(f"{'─' * 80}")
    print(f"Test Query #{i}: \"{query}\"")
    print(f"{'─' * 80}\n")

    # Search in documentation store
    print("Top-3 Results from Documentation Store:")
    doc_results = test_similarity_search(doc_store, query, k=3)

    for j, (doc, score) in enumerate(doc_results, 1):
        print(f"\n  Result #{j} (similarity: {1-score:.4f}):")
        print(f"  Title: {doc.metadata['heading_title']}")
        print(f"  Content preview: {doc.page_content[:150]}...")

    # Search in code store
    print(f"\nTop-3 Results from Code Examples Store:")
    code_results = test_similarity_search(code_store, query, k=3)

    for j, (doc, score) in enumerate(code_results, 1):
        print(f"\n  Result #{j} (similarity: {1-score:.4f}):")
        print(f"  Title: {doc.metadata['heading_title']}")
        print(f"  Summary: {doc.metadata['summary'][:150]}...")
        print(f"  Keywords: {doc.metadata['keywords']}")

    print()


DEMONSTRATION: Similarity Search Tests

────────────────────────────────────────────────────────────────────────────────
Test Query #1: "How do I create binary decision variables for optimization?"
────────────────────────────────────────────────────────────────────────────────

Top-3 Results from Documentation Store:

  Result #1 (similarity: 0.0550):
  Title: Variables
  Content preview: 1.1 Variables Decision variables capture the results of the optimization. In a feasible solution, the computed values for the decision variables satis...

  Result #2 (similarity: 0.0340):
  Title: Binary Variables
  Content preview: 1.1.3 Binary Variables Binary variables are the most constrained variable type that can be added to your model. A binary variable takes a value of eit...

  Result #3 (similarity: -0.0332):
  Title: I Introduction
  Content preview: Gurobi Optimizer Reference Manual This is the manual for version 12.0 of the Gurobi Optimizer. It covers Gurobi’s modeling structures, feat

In [7]:
print(f"{'=' * 80}")
print("STATISTICS")
print(f"{'=' * 80}\n")

print("Documentation Vector Store:")
print(f"  Total vectors: {len(doc_documents)}")
print(f"  Embedding dimension: 384 (all-MiniLM-L6-v2)")
print(f"  Index type: Flat (exact search)")
print(f"  Storage: {VECTORSTORE_DOC_PATH}")

# Estimate storage size
import sys
doc_index_size = os.path.getsize(os.path.join(VECTORSTORE_DOC_PATH, "index.faiss"))
doc_pkl_size = os.path.getsize(os.path.join(VECTORSTORE_DOC_PATH, "index.pkl"))
print(f"  Index size: {doc_index_size / 1024 / 1024:.2f} MB")
print(f"  Metadata size: {doc_pkl_size / 1024 / 1024:.2f} MB\n")

print("Code Examples Vector Store:")
print(f"  Total vectors: {len(code_documents)}")
print(f"  Embedding dimension: 384 (all-MiniLM-L6-v2)")
print(f"  Index type: Flat (exact search)")
print(f"  Storage: {VECTORSTORE_CODE_PATH}")

code_index_size = os.path.getsize(os.path.join(VECTORSTORE_CODE_PATH, "index.faiss"))
code_pkl_size = os.path.getsize(os.path.join(VECTORSTORE_CODE_PATH, "index.pkl"))
print(f"  Index size: {code_index_size / 1024 / 1024:.2f} MB")
print(f"  Metadata size: {code_pkl_size / 1024 / 1024:.2f} MB\n")

print(f"Total Storage: {(doc_index_size + doc_pkl_size + code_index_size + code_pkl_size) / 1024 / 1024:.2f} MB")

STATISTICS

Documentation Vector Store:
  Total vectors: 20
  Embedding dimension: 384 (all-MiniLM-L6-v2)
  Index type: Flat (exact search)
  Storage: /Users/tasnimahmed/Downloads/tutorial/Standalone/tutorial_vectorstore_save
  Index size: 0.03 MB
  Metadata size: 0.03 MB

Code Examples Vector Store:
  Total vectors: 20
  Embedding dimension: 384 (all-MiniLM-L6-v2)
  Index type: Flat (exact search)
  Storage: /Users/tasnimahmed/Downloads/tutorial/Standalone/tutorial_code_save
  Index size: 0.03 MB
  Metadata size: 0.05 MB

Total Storage: 0.14 MB
